# AMEX Enterprise Credit Risk Platform
## Notebook 27 — Phase 2, Problem 4: Delinquency Escalation / Loss Severity — LGD Modeling
### CRISP-DM: Data Preparation + Modeling + Evaluation

Second of 4 notebooks for Problem 4. Computes a real, data-driven **Escalation Severity Score** for every
holdout customer from their actual D_\* delinquency-column `_trend_slope`, `_trend_delta`, and `_last`
(final-statement level) engineered features — already produced, unmodified, by Problem 1's Notebook 04.
Feature weights come from real feature-target correlation measured on the training split only (no holdout
leakage). The score is bucketed into the 3 severity tiers Notebook 26 defined, and validated against the
**real observed default rate** of each tier.

**Honesty boundary (unchanged from Notebook 26):** the severity score, its feature weights, its tier
cutpoints, and the validation default rates are all 100% computed from real data in this run. Only the
dollar LGD value attached to each tier (loaded from Notebook 26's `lgd_policy.json`) is an editable
`ASSUMPTION`.

**Deliverables:** `severity_feature_weights.csv`, `severity_scores_holdout.csv`, `tier_validation_summary.json`,
an inline tier-default-rate chart.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD PROBLEM 1's REAL FEATURES + NOTEBOOK 26's POLICY
# =============================================================================
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Problem 1's Real Features + Notebook 26's Policy")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ROOT = PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
P1_ARTIFACTS = P1_ROOT / "artifacts"
# Legacy pre-rename data-cache folder: the large engineered/raw CSVs were deliberately excluded
# from the Phase 1 folder reorg copy (regenerable, multi-GB) and still live only under this old
# folder name, wherever it currently sits inside Phase1_Foundation. Checked as a fallback below.
P1_LEGACY_ROOT = P1_ROOT.parent / "Problem 1 Credit Default_Probability of Default"
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "04_Problem4_Delinquency_Escalation_Loss_Severity"
ARTIFACTS_DIR = P4_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p4_policy": P4_ROOT / "01_LGD_Policy",
    "p4_modeling": P4_ROOT / "02_LGD_Modeling",
    "p4_validation_deployment": P4_ROOT / "03_Validation_Deployment",
    "p4_reporting_packaging": P4_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
NB04_SUMMARY_PATH = P1_ARTIFACTS / "notebook_04_summary.json"
LGD_POLICY_PATH = PILLAR_DIRS["p4_policy"] / "lgd_policy.json"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB04_SUMMARY_PATH, "Problem 4 needs Problem 1's engineered D_* features -- run Problem 1's Notebook 04 first."),
    (LGD_POLICY_PATH, "run Notebook 26 (this problem's Notebook 1) first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected project folder.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(LGD_POLICY_PATH, "r", encoding="utf-8") as f:
    LGD_POLICY = json.load(f)

RANDOM_SEED = P1_CONFIG["random_seed"]
WARP_THREAD_COUNT = (
    P1_CONFIG.get("resource_limits", {}).get("warp_thread_count")
    or P1_CONFIG.get("warp_thread_count")
    or P1_CONFIG["hardware"]["logical_cores_detected"]
)


def _resolve_engineered_file(filename: str) -> Path:
    """The engineered CSV was deliberately excluded from the Phase 1 folder reorg (large data
    cache, not a portfolio deliverable), so it may live in one of three places. Checked in order,
    most-authoritative first: (1) this problem's own current Feature_Engineering folder -- where a
    NEW run of the reorganized Notebook 04 writes it; (2) the legacy pre-rename data-cache folder,
    where the real large files were confirmed still sitting; (3) the path literally recorded in
    notebook_04_summary.json, in case it points somewhere else still valid. A candidate must be
    larger than 10KB to count -- rules out a stray placeholder/note file of the same name. Only
    if none match does this raise, with an explicit fix."""
    _candidates = [
        P1_ROOT / "Feature_Engineering" / filename,
        P1_LEGACY_ROOT / "Feature_Engineering" / filename,
        Path(NB04_SUMMARY["output_files"][filename]),
    ]
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > 10_000:
            return _c
    raise FileNotFoundError(
        f"{filename} not found (as real data, >10KB) at any checked location:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\nFix: this file is a large data cache that Notebook 04 (re)generates -- it is NOT part of the "
        "portfolio files copied during the Phase 1 folder reorg. Open and run Problem 1's Notebook 04 "
        "(04_feature_engineering.ipynb) from its CURRENT location in Phase1_Foundation, top to bottom, "
        "once. That writes a fresh copy into this problem's own Feature_Engineering folder and updates "
        "notebook_04_summary.json to point at it -- then re-run this notebook."
    )


TRAIN_SPLIT_ENG_PATH = _resolve_engineered_file("train_split_engineered.csv")
TEST_SPLIT_ENG_PATH = _resolve_engineered_file("test_split_engineered.csv")

TIER_ORDER = LGD_POLICY["tier_order"]
LGD_BY_TIER = {v["tier"]: v["lgd"] for v in LGD_POLICY["lgd_by_tier"]["values"]}
KPI_TARGETS = LGD_POLICY["kpi_targets"]

print(f"Severity tiers (from Notebook 26, real)  : {TIER_ORDER}")
print(f"LGD by tier (Notebook 26, ASSUMPTION)    : {LGD_BY_TIER}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
np.random.seed(RANDOM_SEED)
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD REAL D_* ENGINEERED FEATURES (TREND_SLOPE / TREND_DELTA / LAST)
# =============================================================================
_section("SECTION 3: Load Real D_* Engineered Features (trend_slope / trend_delta / last)")

# --- Discover the real D_* engineered feature columns from the file header --
#     (trend_slope/trend_delta are numeric-only by construction in Notebook 04, but
#     _last is computed for EVERY raw D_* column in Notebook 02, including the
#     categorical ones, e.g. D_63/D_64 hold real string status codes like "CR".
#     No column list is hand-typed here -- candidates come from the real header,
#     and any non-numeric column found below is dropped after actually checking
#     its real dtype, not assumed from its name). ---
_header_cols = pl.read_csv(str(TRAIN_SPLIT_ENG_PATH), n_rows=0).columns
_D_CANDIDATE_COLS = sorted(
    c for c in _header_cols
    if c.startswith("D_") and (c.endswith("_trend_slope") or c.endswith("_trend_delta") or c.endswith("_last"))
)
if len(_D_CANDIDATE_COLS) == 0:
    raise RuntimeError("No D_*_trend_slope/_trend_delta/_last columns found in the engineered feature file. "
                        "Fix: re-run Problem 1's Notebook 04.")

_select_cols = ["customer_ID", "target"] + _D_CANDIDATE_COLS
train_pd = pl.read_csv(str(TRAIN_SPLIT_ENG_PATH), columns=_select_cols).to_pandas()
holdout_pd = pl.read_csv(str(TEST_SPLIT_ENG_PATH), columns=_select_cols).to_pandas()

D_FEATURE_COLS = [
    c for c in _D_CANDIDATE_COLS
    if pd.api.types.is_numeric_dtype(train_pd[c]) and pd.api.types.is_numeric_dtype(holdout_pd[c])
]
_dropped_categorical = sorted(set(_D_CANDIDATE_COLS) - set(D_FEATURE_COLS))
if _dropped_categorical:
    print(f"Dropped {len(_dropped_categorical)} non-numeric D_* column(s) (real categorical status "
          f"codes, e.g. D_63/D_64's last-statement value -- not an escalation-severity signal): "
          f"{_dropped_categorical}")
if len(D_FEATURE_COLS) == 0:
    raise RuntimeError("Every candidate D_* column turned out non-numeric. Fix: re-run Problem 1's Notebook 04.")

print(f"Real numeric D_* escalation-signal columns used: {len(D_FEATURE_COLS)} of {len(_D_CANDIDATE_COLS)} candidates")
print(f"Train split (weight fitting)  : {len(train_pd):,} customers")
print(f"Holdout split (score + validate): {len(holdout_pd):,} customers")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: FEATURE WEIGHTS -- REAL CORRELATION WITH TARGET, TRAIN SPLIT ONLY
# =============================================================================
_section("SECTION 4: Feature Weights -- Real Correlation With Target, Train Split Only")

# --- Weights and standardization stats are fit on the TRAIN split only and then
#     frozen -- applying them to the holdout split avoids any leakage, exactly
#     the same discipline Notebook 05 used for its champion model. ---
_target = train_pd["target"].astype(int).to_numpy()
FEATURE_WEIGHTS, FEATURE_DIRECTION, FEATURE_MEAN, FEATURE_STD = {}, {}, {}, {}
_rows = []
for c in D_FEATURE_COLS:
    _col = train_pd[c].to_numpy(dtype=float)
    _valid = ~np.isnan(_col)
    if _valid.sum() < 100:
        continue
    _corr = np.corrcoef(_col[_valid], _target[_valid])[0, 1]
    if np.isnan(_corr) or _corr == 0.0:
        continue
    FEATURE_WEIGHTS[c] = abs(_corr)
    FEATURE_DIRECTION[c] = 1.0 if _corr > 0 else -1.0
    FEATURE_MEAN[c] = float(np.nanmean(_col))
    _std = float(np.nanstd(_col))
    FEATURE_STD[c] = _std if _std > 1e-9 else 1.0
    # NOTE: correlation_with_target/abs_weight_raw below are rounded to 5dp for human-readable display
    # only -- they are NOT used anywhere in the real score computation (NORM_WEIGHTS, built from the
    # unrounded FEATURE_WEIGHTS above, is what _severity_score() actually uses). normalized_weight,
    # mean, and std, however, ARE consumed downstream by Notebook 28's deployed standalone scorer, so
    # they are saved at FULL float precision -- rounding any of the three would silently make the
    # deployed scorer disagree with this notebook's own real output on some real customers (confirmed:
    # rounding normalized_weight to 5dp across ~200+ features accumulated to ~1e-4-level score error,
    # enough to flip a handful of customers whose true score sits close to a tier cutpoint).
    _rows.append({"feature": c, "correlation_with_target": round(float(_corr), 5),
                   "abs_weight_raw": round(abs(_corr), 5), "direction": FEATURE_DIRECTION[c],
                   "mean": FEATURE_MEAN[c], "std": FEATURE_STD[c]})

if not FEATURE_WEIGHTS:
    raise RuntimeError("No D_* feature achieved a non-zero real correlation with target. Fix: investigate "
                        "Problem 1's Notebook 04 output -- this should not happen on the real AMEX dataset.")

_total_w = sum(FEATURE_WEIGHTS.values())
NORM_WEIGHTS = {c: w / _total_w for c, w in FEATURE_WEIGHTS.items()}
for _r in _rows:
    _r["normalized_weight"] = NORM_WEIGHTS[_r["feature"]]  # full precision -- see note above

weights_df = pd.DataFrame(_rows).sort_values("normalized_weight", ascending=False).reset_index(drop=True)
print(f"Features used in severity score (real, non-zero correlation): {len(weights_df)} of {len(D_FEATURE_COLS)}")
# normalized_weight/mean/std are saved at full precision (Notebook 28's deployed scorer consumes them
# directly) -- rounded only for this console printout, never for the saved CSV.
_display_cols = ["feature", "correlation_with_target", "direction", "normalized_weight", "mean", "std"]
print(weights_df[_display_cols].head(10).round(5).to_string(index=False))
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: SEVERITY SCORE -- COMPUTE ON TRAIN (FOR CUTPOINTS) AND HOLDOUT
# =============================================================================
_section("SECTION 5: Severity Score -- Compute on Train (for Cutpoints) and Holdout")


def _severity_score(df: "pd.DataFrame") -> "np.ndarray":
    score = np.zeros(len(df), dtype=float)
    for c, w in NORM_WEIGHTS.items():
        z = (df[c].fillna(FEATURE_MEAN[c]).to_numpy(dtype=float) - FEATURE_MEAN[c]) / FEATURE_STD[c]
        score += w * FEATURE_DIRECTION[c] * z
    return score


train_severity = _severity_score(train_pd)
holdout_severity = _severity_score(holdout_pd)
holdout_pd = holdout_pd.copy()
holdout_pd["severity_score"] = holdout_severity

# --- ASSUMPTION (per Notebook 26's policy): equal-population tertile cutpoints,
#     fit on the TRAIN split and frozen before being applied to the holdout --
#     no cutpoint is chosen by looking at holdout outcomes. ---
CUT_LOW, CUT_HIGH = np.nanpercentile(train_severity, [33.333, 66.667])
print(f"Tertile cutpoints (fit on train, real)  : low<={CUT_LOW:.4f} < moderate<={CUT_HIGH:.4f} < severe")


def _assign_tier(score: float) -> str:
    if score <= CUT_LOW:
        return TIER_ORDER[0]
    elif score <= CUT_HIGH:
        return TIER_ORDER[1]
    return TIER_ORDER[2]


holdout_pd["severity_tier"] = holdout_pd["severity_score"].apply(_assign_tier)
holdout_pd["lgd_assigned"] = holdout_pd["severity_tier"].map(LGD_BY_TIER)
print(holdout_pd["severity_tier"].value_counts().reindex(TIER_ORDER).to_string())
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: VALIDATION -- REAL OBSERVED DEFAULT RATE BY TIER
# =============================================================================
_section("SECTION 6: Validation -- Real Observed Default Rate by Tier")

_total_defaulters = int(holdout_pd["target"].sum())
_tier_stats = []
for _t in TIER_ORDER:
    _sub = holdout_pd[holdout_pd["severity_tier"] == _t]
    _n = len(_sub)
    _n_def = int(_sub["target"].sum())
    _tier_stats.append({
        "tier": _t,
        "population": _n,
        "population_pct_of_holdout": round(100.0 * _n / len(holdout_pd), 2),
        "defaulters_in_tier": _n_def,
        "pct_of_all_defaulters": round(100.0 * _n_def / _total_defaulters, 2) if _total_defaulters else 0.0,
        "observed_default_rate": round(_n_def / _n, 6) if _n else 0.0,
        "lgd_assigned": LGD_BY_TIER[_t],
    })
tier_validation_df = pd.DataFrame(_tier_stats)
print(tier_validation_df.to_string(index=False))

_rates = [r["observed_default_rate"] for r in _tier_stats]
_is_monotonic = all(_rates[i] < _rates[i + 1] for i in range(len(_rates) - 1))
_ratio = (_rates[-1] / _rates[0]) if _rates[0] > 0 else float("inf")
_min_tier_defaulter_share = min(r["pct_of_all_defaulters"] for r in _tier_stats)

_checks_passed = True


def _check(label, condition, detail="", hard=True):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        if hard:
            _checks_passed = False
        _mark = "\u274c" if hard else "\u26a0\ufe0f"
        print(f"{_mark} {label}  {detail}")


_check("Real observed default rate strictly increases Low -> Moderate -> Severe", _is_monotonic,
       f"(rates={_rates})")
_check(f"Severe/Low default-rate ratio meets target ({KPI_TARGETS['min_default_rate_ratio_top_to_bottom_tier']}x)",
       _ratio >= KPI_TARGETS["min_default_rate_ratio_top_to_bottom_tier"], f"(measured={_ratio:.2f}x)", hard=False)
_check(f"Every tier holds >= {KPI_TARGETS['min_tier_population_pct']}% of real defaulters",
       _min_tier_defaulter_share >= KPI_TARGETS["min_tier_population_pct"],
       f"(min measured={_min_tier_defaulter_share:.2f}%)", hard=False)

print("\n(Monotonicity is a hard requirement -- the whole point of this notebook. The ratio and population-"
      "balance targets are ASSUMPTION-set KPI goals: measured real values are reported above whether or not "
      "they clear the goal, per the no-fabrication rule.)")

if not _checks_passed:
    raise RuntimeError("Severity tiers do not rank-order real observed default rates monotonically -- "
                        "the tier-differentiated LGD cannot be justified. See \u274c line above.")

print("\nAll hard verification checks passed.")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: INLINE CHART -- REAL OBSERVED DEFAULT RATE BY SEVERITY TIER
# =============================================================================
_section("SECTION 7: Inline Chart -- Real Observed Default Rate by Severity Tier")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "surface": "#FFFFFF"}
fig, ax = plt.subplots(figsize=(7.5, 5), dpi=150)
_bars = ax.bar(TIER_ORDER, _rates, color=[VIZ["muted"], VIZ["accent"], VIZ["ink"]])
for _b, _r in zip(_bars, _rates):
    ax.text(_b.get_x() + _b.get_width() / 2, _r, f"{_r:.2%}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Real observed default rate (holdout)")
ax.set_title("Problem 4: Escalation Severity Tier vs Real Observed Default Rate")
fig.tight_layout()
chart_path = PILLAR_DIRS["p4_modeling"] / "severity_tier_default_rate_chart.png"
fig.savefig(chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig)
print(f"\u2705 Saved -> {chart_path.name} (this problem's 02_LGD_Modeling folder)")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: SAVE ARTIFACTS
# =============================================================================
_section("SECTION 8: Save Artifacts")

weights_path = PILLAR_DIRS["p4_modeling"] / "severity_feature_weights.csv"
weights_df.to_csv(weights_path, index=False)

scores_path = PILLAR_DIRS["p4_modeling"] / "severity_scores_holdout.csv"
holdout_pd[["customer_ID", "severity_score", "severity_tier", "target", "lgd_assigned"]].to_csv(
    scores_path, index=False)

validation_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_features_used": int(len(weights_df)),
    "cutpoints": {"low_upper_bound": float(CUT_LOW), "moderate_upper_bound": float(CUT_HIGH)},
    "tier_stats": _tier_stats,
    "monotonic": bool(_is_monotonic),
    "severe_to_low_ratio": float(_ratio),
    "min_tier_defaulter_share_pct": float(_min_tier_defaulter_share),
    "kpi_targets": KPI_TARGETS,
}
validation_path = PILLAR_DIRS["p4_modeling"] / "tier_validation_summary.json"
with open(validation_path, "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, indent=2)

_expected_files = [weights_path, scores_path, validation_path, chart_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)
if not _checks_passed:
    raise RuntimeError("One or more Notebook 27 output files failed to save. See \u274c line above.")

for fp in _expected_files:
    print(f"\u2705 Saved -> {fp.name} (this problem's 02_LGD_Modeling folder)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE NOTEBOOK 27 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 9: Write Notebook 27 Summary Artifact")

notebook_27_summary = {
    "notebook": "27_lgd_modeling", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 4, "problem_name": "Delinquency Escalation / Loss Severity",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "n_holdout_customers": int(len(holdout_pd)),
    "n_features_used": int(len(weights_df)),
    "tier_order": TIER_ORDER, "lgd_by_tier": LGD_BY_TIER,
    "monotonic_validated": bool(_is_monotonic),
    "severe_to_low_default_rate_ratio": float(_ratio),
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb27_summary_path = ARTIFACTS_DIR / "notebook_27_summary.json"
with open(nb27_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_27_summary, f, indent=2)
print(f"\u2705 Saved -> {nb27_summary_path.name} (this problem's artifacts folder)")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 10: Notebook 27 Complete -- Handoff to Notebook 28")

print("NOTEBOOK 27: LGD MODELING -- COMPLETE")
print(f"  Holdout customers scored            : {len(holdout_pd):,}")
print(f"  Real D_* features used              : {len(weights_df)}")
print(f"  Tier rank-ordering validated (real) : {_is_monotonic}")
print(f"  Severe/Low default-rate ratio (real): {_ratio:.2f}x")
print(f"  Files produced                      : {len(_expected_files) + 1}")
for _p in _expected_files + [nb27_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                       : 28_validation_deployment.ipynb")
print("\n\u2705 Ready to proceed.")
